In [1]:
# conda vis2
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import at3d
import make_animation as ma
import proj_to_altitude as proj

PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

plt.style.use('seaborn-v0_8-poster')

In [2]:
import src.Retrieval as retrieval

In [3]:
EXPERIMENT = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V11' / 'default'

r = retrieval.Retrieval(EXPERIMENT)
r.load_retrieval()

In [4]:
sensor_dict = at3d.containers.SensorsDict()
zenith = np.arange(5, 71, 5)
zenith = np.append(np.arange(70, -1, -5), np.arange(5, 71, 5))
n = len(zenith)
azimuth = np.append(-180 * np.ones(int((n-1)/2)), np.zeros(int((n-1)/2 + 1)))
assert len(zenith) == len(azimuth)

for z, a in zip(zenith, azimuth):
    sensor_dict.add_sensor('flyover', at3d.sensor.orthographic_projection(
        wavelength = 0.67,
        bounding_box = r.rte_grid,
        x_resolution = 0.1,
        y_resolution = 0.1,
        azimuth = a,
        zenith = z,
        altitude = 'TOA',
        stokes = 'I'
    ))

In [5]:
outfile = './rte_solution_V11_default.nc'
r.solvers_retrieval[0.67].load_solution(outfile)

TypeError: string indices must be integers

In [ ]:
sensor_dict.get_measurements(r.solvers_retrieval)

In [ ]:
outfile = './rte_solution_V11_default.nc'
solution = r.solvers_retrieval[0.67].save_solution()

In [ ]:
solution.to_netcdf(outfile)

In [ ]:
def plot_frame(cam, ax=None, plot_kwargs={}):
    if(ax is None):
        f, ax = plt.subplots(1, 1, figsize=(7,5))
    c = ax.tricontourf(cam.cam_x, cam.cam_y, cam.I, **plot_kwargs)
    plt.colorbar(c, ax=ax, shrink=0.8)
    if(cam.projection == 'Orthographic'):
        ax.set_title(f'Zenith: {cam.projection_zenith:.1f}\nAzimuth: {cam.projection_azimuth:.1f}')
    
    return f, ax

In [51]:
plot_kwargs = {
    'cmap':'gray',
    'vmin':0,
    'vmax':0.2
}
plot_kwargs['levels'] = np.linspace(plot_kwargs['vmin'], plot_kwargs['vmax'], 50)

# cam = sensor_dict['flyover']['sensor_list'][0]
# f, ax = plot_frame(cam, plot_kwargs=plot_kwargs)

for i, cam in enumerate(sensor_dict['flyover']['sensor_list']):
    f, ax = plot_frame(proj.project_sensor_to_altitude(cam, 0.0), plot_kwargs=plot_kwargs)
    ax.set_xlim([0, 17.6])
    ax.set_ylim([0, 17.6])
    f.savefig(f'./flyover/frame{i:02d}')
    plt.close(f)

In [55]:
ma.create_animation(frame_dir='./flyover/', output_path='./flyover00.webp', fmt='webp')

TypeError: create_animation() got an unexpected keyword argument 'fmt'